# Notebook: 01_modulised_data_preparation.ipynb

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

import os

os.chdir("/Users/borisivanov/Documents/GitHub/sales-management-project-op-analytics")
print("Now in:", os.getcwd())

# LOAD MODULES (now much cleaner!)
from modules.load.load_data import load_raw_data
from modules.extract.extract_basket import parse_basket
from modules.extract.extract_order_header import build_order_header
from modules.extract.extract_line_items import extract_all_line_items

from modules.transform.clean_orders import clean_orders
from modules.transform.currency import add_fx_rates, convert_amounts_to_eur
from modules.transform.sessions import build_sessions_dim, compute_conversion_flags
from modules.transform.funnel import build_funnel, calculate_funnel_metrics, funnel_by_dimension
from modules.transform.exit_events import extract_exit_events, identify_exit_sessions, summarize_exit_points

from modules.attribution.attribution import (
    prepare_dm_clicks,
    build_last_click_table,
    apply_attribution_window,
    finalize_attribution,
    attribution_summary
)

from modules.transform.repeat_buyers import (
    identify_customers,
    classify_orders,
    calculate_repeat_metrics,
    build_cohort_analysis
)

from modules.output.export import export_tables
from modules.config import FX_RATES


/Users/borisivanov/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Now in: /Users/borisivanov/Documents/GitHub/sales-management-project-op-analytics


***
# Load Data

In [2]:
orders_o50, orders_h50, abandoned, clicks_app, clicks_dm = load_raw_data()

/Users/borisivanov/Documents/GitHub/sales-management-project-op-analytics/modules/load/load_data.py:13: DtypeWarning: Columns (10,15) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(DATA_RAW / "op-app_clicks.csv")


***
# Orders H50

In [3]:
orders_h50.head()

,id,createdate,purchaseid,status,rcv_email,processed,shipped,cust_email,modification_date,seo_processed,followup,basket,delivered,externalid,modified_date
0,157397,2025-09-01 13:46:04.79125+02,iyxumf11yzxl,COMPLETED,op@originalpeople.com,True,NaN,anon+5fed413d9994@example.invalid,2025-09-01 13:46:04.79125+02,NaN,False,"{""memo"": ""Originalpeople"", ""domain"": ""original...",False,NaN,2025-09-22 18:01:07.275005+02
1,157396,2025-09-01 13:13:36.19506+02,iyxumf10r2h3,COMPLETED,op@originalpeople.com,True,NaN,anon+206f609a1079@example.invalid,2025-09-01 13:13:36.19506+02,NaN,False,"{""memo"": ""Originalpeople"", ""domain"": ""original...",False,NaN,2025-09-22 18:01:07.275005+02
2,157395,2025-09-01 12:50:49.825583+02,iyxumf100s7h,COMPLETED,op@originalpeople.com,True,NaN,anon+651cf69e7e83@example.invalid,2025-09-01 12:50:49.825583+02,NaN,False,"{""memo"": ""Originalpeople"", ""domain"": ""original...",False,NaN,2025-09-22 18:01:07.275005+02
3,157394,2025-09-01 11:41:22.378909+02,iyxumf0xivfg,COMPLETED,op@originalpeople.com,True,NaN,anon+4eea1d4a3644@example.invalid,2025-09-01 11:41:22.378909+02,NaN,False,"{""memo"": ""Originalpeople"", ""domain"": ""original...",False,NaN,2025-09-22 18:01:07.275005+02
4,157393,2025-09-01 09:49:37.827571+02,7n8qmf0tjrgk,COMPLETED,op@originalpeople.com,True,NaN,anon+2684c811fc3e@example.invalid,2025-09-01 09:49:37.827571+02,NaN,False,"{""memo"": ""Originalpeople"", ""domain"": ""original...",False,NaN,2025-09-22 18:01:07.275005+02


***
# Orders O50

In [4]:
orders_o50.head()

,id,createdate,purchaseid,status,rcv_email,processed,shipped,cust_email,modification_date,seo_processed,followup,basket,delivered,externalid,modified_date
0,105554,2021-11-08 16:53:12.712285+01,4z5okvqug0ql,COMPLETED,op@originalpeople.com,True,2021-11-09 12:09:28.510493+01,anon+9257a74fcd39@example.invalid,2021-11-08 16:53:12.712285+01,NaN,True,"{""memo"": ""Originalpeople"", ""locale"": ""es-ES"", ...",True,NaN,2025-09-22 18:00:46.373704+02
1,105553,2021-11-08 16:40:10.811983+01,4z5okvqtz9je,COMPLETED,op@originalpeople.com,True,2021-11-10 12:19:24.822968+01,anon+b87a237218de@example.invalid,2021-11-08 16:40:10.811983+01,NaN,True,"{""memo"": ""Originalpeople"", ""locale"": ""es-ES"", ...",True,NaN,2025-09-22 18:00:46.373704+02
2,105552,2021-11-08 16:38:06.417345+01,4z5okvqtwlhx,COMPLETED,op@originalpeople.com,True,2021-11-10 13:03:46.525788+01,anon+fa1f444136e5@example.invalid,2021-11-08 16:38:06.417345+01,NaN,True,"{""memo"": ""Originalpeople"", ""locale"": ""fr-FR"", ...",True,NaN,2025-09-22 18:00:46.373704+02
3,105551,2021-11-08 16:03:13.837937+01,4z5okvqsnruu,COMPLETED,op@originalpeople.com,True,2021-11-10 12:19:30.692814+01,anon+0728685568f7@example.invalid,2021-11-08 16:03:13.837937+01,NaN,True,"{""memo"": ""Originalpeople"", ""locale"": ""es-ES"", ...",True,NaN,2025-09-22 18:00:46.373704+02
4,105550,2021-11-08 15:57:21.566287+01,4z5okvqsg7lj,COMPLETED,op@originalpeople.com,True,2021-11-10 12:19:36.697581+01,anon+76a418f44283@example.invalid,2021-11-08 15:57:21.566287+01,NaN,True,"{""memo"": ""Originalpeople"", ""locale"": ""es-ES"", ...",True,NaN,2025-09-22 18:00:46.373704+02


In [5]:
# Check start and end periods of both datasets
print("O50K Dataset:")
print("Start Date:",orders_o50['createdate'].min())
print("End Date:", orders_o50['createdate'].max())

print("\nH50K Dataset:")
print("Start Date:",orders_h50['createdate'].min())
print("End Date:", orders_h50['createdate'].max())

O50K Dataset:
Start Date: 2016-09-26 15:48:37.71034+02
End Date: 2024-10-27 22:00:00+01

H50K Dataset:
Start Date: 2021-11-08 17:07:45.983884+01
End Date: 2025-09-01 13:46:04.79125+02


***
# Abandoned 

In [6]:
abandoned.head()

,sessionid,total,email,processed,basket_page,payment_page,pay_intent,processor,language,country,...,createdate,currency,id,comment,design_page,system_userneedshelp,session_page,modifydate,reminder,newsletter
0,1o1mniiyxumf0zom9s,31.6,anon+845f3dca262a@example.invalid,True,True,True,True,paycomet,es,ES,...,2025-09-01 13:07:31.586762+02,EUR,79740,NaN,False,False,False,2025-09-01 13:07:31.586762+02,True,NaN
1,1o1mniiyxumf10529y,NaN,anon+161a1021cfb5@example.invalid,True,False,False,False,NaN,es,ES,...,2025-09-01 13:01:06.893331+02,EUR,79739,NaN,True,False,False,2025-09-01 13:01:06.893331+02,False,NaN
2,1o1mni7n8qmeya8kc0,118.0,anon+d537ca779eab@example.invalid,True,True,True,True,braintree,sv,SE,...,2025-09-01 12:48:32.838503+02,SEK,79738,NaN,False,False,False,2025-09-01 12:48:32.838503+02,True,NaN
3,1o1mni7n8qmevcswbn,17.9,NaN,False,True,True,False,NaN,en,GB,...,2025-09-01 12:17:57.02138+02,GBP,79737,NaN,False,False,False,2025-09-01 12:17:57.02138+02,False,NaN
4,1o1mniiyxumf0yofri,0.0,NaN,False,True,False,False,NaN,en,RO,...,2025-09-01 12:13:12.715813+02,EUR,79736,NaN,False,False,False,2025-09-01 12:13:12.715813+02,False,NaN


In [7]:
abandoned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2558 entries, 0 to 2557
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   sessionid             2558 non-null   object 
 1   total                 1977 non-null   float64
 2   email                 1276 non-null   object 
 3   processed             2558 non-null   bool   
 4   basket_page           2558 non-null   bool   
 5   payment_page          2558 non-null   bool   
 6   pay_intent            2558 non-null   bool   
 7   processor             735 non-null    object 
 8   language              2558 non-null   object 
 9   country               2558 non-null   object 
 10  url_base              2558 non-null   object 
 11  createdate            2558 non-null   object 
 12  currency              2558 non-null   object 
 13  id                    2558 non-null   int64  
 14  comment               2 non-null      object 
 15  design_page          

In [8]:
# Check for missing values in the whole dataframe
abandoned.isnull().sum()

# turn to percentages
abandoned.isnull().mean() * 100

sessionid                0.000000
total                   22.713057
email                   50.117279
processed                0.000000
basket_page              0.000000
payment_page             0.000000
pay_intent               0.000000
processor               71.266615
language                 0.000000
country                  0.000000
url_base                 0.000000
createdate               0.000000
currency                 0.000000
id                       0.000000
comment                 99.921814
design_page              0.000000
system_userneedshelp     0.000000
session_page             0.000000
modifydate               0.000000
reminder                 0.000000
newsletter              99.921814
dtype: float64

***
# Clicks App

In [9]:
clicks_app.head()

,id,date,sid,txt,type,data,language,country,data_id,release_id,host,domain,page,screen_h,screen_w,url
0,966106,2025-09-01 14:22:24.100075+02,1o1mniiyxumf139kqv,personlichen-aufkleber-erstellen.html?whichBui...,A,"{""source"": {""desc"": ""personlichen-aufkleber-er...",de,DE,card-car-sticker,1756713179060,NaN,originalpeople.com,stickers.html,780.0,360.0,/de/personalised-custom-car-stickers.html
1,966105,2025-09-01 14:20:23.552809+02,1o1mniiyxumf136pau,main-menu-stickers,DIV,"{""source"": {""desc"": ""main-menu-stickers"", ""typ...",fr,FR,main-menu-stickers,1756713179060,NaN,originalpeople.com,blogs/decore-ta-boite-aux-lettres-stickers-per...,735.0,339.0,/fr/blogs/decore-ta-boite-aux-lettres-stickers...
2,966104,2025-09-01 14:20:18.53399+02,1o1mni7n8qmel905u7,/images/svg/header-globe-white.svg,IMG,"{""source"": {""alt"": """", ""src"": ""/images/svg/hea...",es,ES,main-menu-store,1756713179060,NaN,originalpeople.es,companies.html,720.0,1280.0,/companies.html
3,966103,2025-09-01 14:20:10.770718+02,1o1mni7n8qmel905u7,/companies.html,A,"{""source"": {""desc"": ""/companies.html"", ""href"":...",es,ES,main-menu-company,1756713179060,NaN,originalpeople.es,catalogue.html,720.0,1280.0,/todos-los-productos.html
4,966102,2025-09-01 14:17:15.891942+02,1o1mni7n8qmel905u7,main-menu-packsofsymbols,LI,"{""source"": {""desc"": ""main-menu-packsofsymbols""...",es,ES,main-menu-packsofsymbols,1756713179060,NaN,originalpeople.es,catalogue.html,720.0,1280.0,/todos-los-productos.html


In [10]:
clicks_app.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 965961 entries, 0 to 965960
Data columns (total 16 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   id          965961 non-null  int64  
 1   date        965961 non-null  object 
 2   sid         965961 non-null  object 
 3   txt         965956 non-null  object 
 4   type        965956 non-null  object 
 5   data        965961 non-null  object 
 6   language    965961 non-null  object 
 7   country     965961 non-null  object 
 8   data_id     965961 non-null  object 
 9   release_id  965961 non-null  int64  
 10  host        523 non-null     object 
 11  domain      953060 non-null  object 
 12  page        953060 non-null  object 
 13  screen_h    953060 non-null  float64
 14  screen_w    953060 non-null  float64
 15  url         878160 non-null  object 
dtypes: float64(2), int64(2), object(12)
memory usage: 117.9+ MB


In [11]:
# Check for missing values in the whole dataframe
clicks_app.isnull().sum()

# turn to percentages
clicks_app.isnull().mean() * 100

id             0.000000
date           0.000000
sid            0.000000
txt            0.000518
type           0.000518
data           0.000000
language       0.000000
country        0.000000
data_id        0.000000
release_id     0.000000
host          99.945857
domain         1.335561
page           1.335561
screen_h       1.335561
screen_w       1.335561
url            9.089497
dtype: float64

***
# Clicks DM

In [12]:
clicks_dm.head()

,date,domain,sid,platform,url,ad,ad_type,ad_name,attr_value,attr_items,purchaseid,id,ad_country,ad_extension
0,2025-09-01 13:53:36+02,originalpeople.com,1o1mniiyxumf129k8j,GA,/de/personalised-custom-car-stickers.html,de.shopping.caravan-feeds.caravan.de.4,shopping,caravan-feeds,NaN,NaN,NaN,874454,de,caravan.de.4
1,2025-09-01 13:53:39+02,originalpeople.com,1o1mni7n8qmf0gf8cu,GA,/es/pegatinas-personalizadas.html,es.shopping.caravan-feeds.caravan.es.4,shopping,caravan-feeds,NaN,NaN,NaN,874453,es,caravan.es.4
2,2025-09-01 13:52:07+02,originalpeople.com,1o1mniiyxumf125w46,GA,/de/personalised-custom-car-stickers.html,de.shopping.caravan-feeds.caravan.de.2,shopping,caravan-feeds,NaN,NaN,NaN,874452,de,caravan.de.2
3,2025-09-01 13:54:38+02,originalpeople.com,1o1mniiyxumf12avv3,GA,/es/,es.discovery.gifts,discovery,gifts,NaN,NaN,NaN,874451,es,NaN
4,2025-09-01 13:54:36+02,originalpeople.com,1o1mniiyxumf12au5g,FB,/de/personalised-custom-car-stickers.html,de.carousel.caravan,carousel,caravan,NaN,NaN,NaN,874450,de,NaN


In [13]:
clicks_dm.columns

Index(['date', 'domain', 'sid', 'platform', 'url', 'ad', 'ad_type', 'ad_name',
       'attr_value', 'attr_items', 'purchaseid', 'id', 'ad_country',
       'ad_extension'],
      dtype='object')

In [14]:
clicks_dm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 213884 entries, 0 to 213883
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   date          213884 non-null  object 
 1   domain        213884 non-null  object 
 2   sid           213884 non-null  object 
 3   platform      213884 non-null  object 
 4   url           213884 non-null  object 
 5   ad            213884 non-null  object 
 6   ad_type       213884 non-null  object 
 7   ad_name       201321 non-null  object 
 8   attr_value    7071 non-null    float64
 9   attr_items    7071 non-null    object 
 10  purchaseid    7071 non-null    object 
 11  id            213884 non-null  int64  
 12  ad_country    213363 non-null  object 
 13  ad_extension  126335 non-null  object 
dtypes: float64(1), int64(1), object(12)
memory usage: 22.8+ MB


In [15]:
# Check for missing values in the whole dataframe
clicks_dm.isnull().sum()

# turn to percentages
clicks_dm.isnull().mean() * 100

date             0.000000
domain           0.000000
sid              0.000000
platform         0.000000
url              0.000000
ad               0.000000
ad_type          0.000000
ad_name          5.873745
attr_value      96.694002
attr_items      96.694002
purchaseid      96.694002
id               0.000000
ad_country       0.243590
ad_extension    40.932936
dtype: float64

***
# Clean & merge orders (o50 + h50)

In [16]:
orders_raw = clean_orders(orders_o50, orders_h50)

In [17]:
orders_raw.head()

,id,createdate,purchaseid,status,rcv_email,processed,shipped,cust_email,modification_date,seo_processed,followup,basket,delivered,modified_date,source,is_shipped
0,1160,2016-09-26 15:48:37.71034+02,BT-f7kq5hcd,COMPLETED,op@originalpeople.com,True,2016-09-26 15:54:35.932894+02,anon+9a697ab42d37@example.invalid,2016-09-26 15:48:37.71034+02,2018-05-10,False,"{""memo"": ""Originalpeople"", ""locale"": ""es-ES"", ...",False,2025-09-22 18:00:21.313321+02,o50k,True
1,1161,2016-09-26 15:51:47.935259+02,PAY-2FY53479L06789303K7USP4Y,COMPLETED,op@originalpeople.com,True,2016-09-26 15:54:35.922434+02,anon+9a697ab42d37@example.invalid,2016-09-26 15:51:47.935259+02,2018-05-10,False,"{""memo"": ""Originalpeople"", ""locale"": ""es-ES"", ...",False,2025-09-22 18:00:21.313321+02,o50k,True
2,1162,2016-09-26 20:19:39.710063+02,PAY-8B5571432E8625602K7UWNOY,CREATED,op@originalpeople.com,False,NaN,anon+bb33103e784d@example.invalid,2016-09-26 20:19:39.710063+02,2018-05-10,False,"{""memo"": ""Originalpeople"", ""locale"": ""en-US"", ...",False,2025-09-22 18:00:21.313321+02,o50k,False
3,1163,2016-09-26 20:25:35.678834+02,PAY-3N2137657L732135KK7UWQHY,CREATED,op@originalpeople.com,False,NaN,anon+bb33103e784d@example.invalid,2016-09-26 20:25:35.678834+02,2018-05-10,False,"{""memo"": ""Originalpeople"", ""locale"": ""en-US"", ...",False,2025-09-22 18:00:21.313321+02,o50k,False
4,1164,2016-09-26 20:26:53.532779+02,PAY-81N39817L54501647K7UWQ3I,CREATED,op@originalpeople.com,False,NaN,anon+7f013fb95025@example.invalid,2016-09-26 20:26:53.532779+02,2018-05-10,False,"{""memo"": ""Originalpeople"", ""locale"": ""en-US"", ...",False,2025-09-22 18:00:21.313321+02,o50k,False


***
# Parse Basket

In [18]:
orders_raw = parse_basket(orders_raw)

In [19]:
orders_raw.head()

,id,createdate,purchaseid,status,rcv_email,processed,shipped,cust_email,modification_date,seo_processed,followup,basket,delivered,modified_date,source,is_shipped,basket_parsed
0,1160,2016-09-26 15:48:37.71034+02,BT-f7kq5hcd,COMPLETED,op@originalpeople.com,True,2016-09-26 15:54:35.932894+02,anon+9a697ab42d37@example.invalid,2016-09-26 15:48:37.71034+02,2018-05-10,False,"{""memo"": ""Originalpeople"", ""locale"": ""es-ES"", ...",False,2025-09-22 18:00:21.313321+02,o50k,True,"{'memo': 'Originalpeople', 'locale': 'es-ES', ..."
1,1161,2016-09-26 15:51:47.935259+02,PAY-2FY53479L06789303K7USP4Y,COMPLETED,op@originalpeople.com,True,2016-09-26 15:54:35.922434+02,anon+9a697ab42d37@example.invalid,2016-09-26 15:51:47.935259+02,2018-05-10,False,"{""memo"": ""Originalpeople"", ""locale"": ""es-ES"", ...",False,2025-09-22 18:00:21.313321+02,o50k,True,"{'memo': 'Originalpeople', 'locale': 'es-ES', ..."
2,1162,2016-09-26 20:19:39.710063+02,PAY-8B5571432E8625602K7UWNOY,CREATED,op@originalpeople.com,False,NaN,anon+bb33103e784d@example.invalid,2016-09-26 20:19:39.710063+02,2018-05-10,False,"{""memo"": ""Originalpeople"", ""locale"": ""en-US"", ...",False,2025-09-22 18:00:21.313321+02,o50k,False,"{'memo': 'Originalpeople', 'locale': 'en-US', ..."
3,1163,2016-09-26 20:25:35.678834+02,PAY-3N2137657L732135KK7UWQHY,CREATED,op@originalpeople.com,False,NaN,anon+bb33103e784d@example.invalid,2016-09-26 20:25:35.678834+02,2018-05-10,False,"{""memo"": ""Originalpeople"", ""locale"": ""en-US"", ...",False,2025-09-22 18:00:21.313321+02,o50k,False,"{'memo': 'Originalpeople', 'locale': 'en-US', ..."
4,1164,2016-09-26 20:26:53.532779+02,PAY-81N39817L54501647K7UWQ3I,CREATED,op@originalpeople.com,False,NaN,anon+7f013fb95025@example.invalid,2016-09-26 20:26:53.532779+02,2018-05-10,False,"{""memo"": ""Originalpeople"", ""locale"": ""en-US"", ...",False,2025-09-22 18:00:21.313321+02,o50k,False,"{'memo': 'Originalpeople', 'locale': 'en-US', ..."


In [20]:
# Check for missing values in the whole dataframe
orders_raw.isnull().sum()

# turn to percentages
orders_raw.isnull().mean() * 100

id                    0.000000
createdate            0.000000
purchaseid            0.000000
status                0.000000
rcv_email             0.000000
processed             0.000752
shipped               5.605403
cust_email            0.000000
modification_date     0.000000
seo_processed        46.328623
followup              0.000000
basket                0.000000
delivered             0.000000
modified_date         0.000000
source                0.000000
is_shipped            0.000000
basket_parsed         0.000000
dtype: float64

***
# Extract Order Header

In [21]:
order_header_df = build_order_header(orders_raw)

In [22]:
order_header_df.head()

,order_id,purchaseid,createdate,status,sessionid,tracking_id,currency,language,locale,country,...,sender_firstname,sender_lastname,invoice_fee,processed,shipped,delivered,cust_email,is_shipped,fx_rate,amount_eur
0,1160,BT-f7kq5hcd,2016-09-26 15:48:37.71034+02,COMPLETED,None,efpEd0Xi7FHS5rXtNTXCOQq68fSlM2oT,EUR,es,es-ES,ES,...,Name,Surname,0,True,2016-09-26 15:54:35.932894+02,False,anon+9a697ab42d37@example.invalid,True,1.00,13.00
1,1161,PAY-2FY53479L06789303K7USP4Y,2016-09-26 15:51:47.935259+02,COMPLETED,None,efpEd0Xi7FHS5rXtNTXCOQq68fSlM2oT,EUR,es,es-ES,ES,...,Name,Surname,0,True,2016-09-26 15:54:35.922434+02,False,anon+9a697ab42d37@example.invalid,True,1.00,5.00
2,1162,PAY-8B5571432E8625602K7UWNOY,2016-09-26 20:19:39.710063+02,CREATED,None,DXoGQ8UFEPzInxEgnyR2bSuEIzPrpSDc,USD,en,en-US,US,...,Name,Surname,0,False,NaN,False,anon+bb33103e784d@example.invalid,False,0.92,5.98
3,1163,PAY-3N2137657L732135KK7UWQHY,2016-09-26 20:25:35.678834+02,CREATED,None,DXoGQ8UFEPzInxEgnyR2bSuEIzPrpSDc,USD,en,en-US,US,...,Name,Surname,0,False,NaN,False,anon+bb33103e784d@example.invalid,False,0.92,5.98
4,1164,PAY-81N39817L54501647K7UWQ3I,2016-09-26 20:26:53.532779+02,CREATED,None,DXoGQ8UFEPzInxEgnyR2bSuEIzPrpSDc,USD,en,en-US,US,...,Name,Surname,0,False,NaN,False,anon+7f013fb95025@example.invalid,False,0.92,5.98


In [23]:
# Check for missing values in the whole dataframe
order_header_df.isnull().sum()

# turn to percentages
order_header_df.isnull().mean() * 100

order_id             0.000000
purchaseid           0.000000
createdate           0.000000
status               0.000000
sessionid            2.667700
tracking_id         97.333053
currency             0.000000
language             0.000000
locale               0.001504
country              0.000000
referrer             0.001504
ship_name            0.003008
ship_surname         0.003008
ship_address         0.003008
ship_address2        0.003008
ship_country         0.003008
ship_province        0.003008
ship_postalcode      0.003008
ship_city            0.003008
rcv_email            0.000000
order_amount         0.000000
sender_email         0.000752
sender_firstname     0.003008
sender_lastname      0.003008
invoice_fee         97.333053
processed            0.000752
shipped              5.605403
delivered            0.000000
cust_email           0.000000
is_shipped           0.000000
fx_rate              0.000000
amount_eur           0.000000
dtype: float64

***
# Extract Line Items

In [24]:
order_line_items_df = extract_all_line_items(orders_raw)
order_line_items_df.head()

,order_id,sku,description,qty,unit_price,tax_percentage,currency,color,size,font,img_label,inverted,align,subtotal,unit_price_eur,subtotal_eur
0,1160,100010,"Blanco, 120 mm de altura - XAVI Y FRED",5,1.818182,0.21,EUR,Blanco,120,Alegreya,XAVI Y FRED,False,center,9.090909,1.818182,9.090909
1,1161,100010,"Blanco, 120 mm de altura - ORIGINALPEOPLE",1,2.479339,0.21,EUR,Blanco,120,Alegreya,ORIGINALPEOPLE,False,left,2.479339,2.479339,2.479339
2,1162,100010,"White, 4.7in high - ORIGINALPEOPLE",1,3.000000,0.00,USD,White,120,Alegreya,ORIGINALPEOPLE,False,center,3.000000,2.760000,2.760000
3,1163,100010,"White, 4.7in high - ORIGINALPEOPLE",1,3.000000,0.00,USD,White,120,Alegreya,ORIGINALPEOPLE,False,center,3.000000,2.760000,2.760000
4,1164,100010,"White, 4.7in high - ORIGINALPEOPLE",1,3.000000,0.00,USD,White,120,Alegreya,ORIGINALPEOPLE,False,center,3.000000,2.760000,2.760000


***
# BUILD SESSIONS

In [26]:
sessions_dim = build_sessions_dim(abandoned)
sessions_dim.head()

,sessionid,email,createdate,language,country,processed,total
0,1o1mni21lcmdmzyowk,anon+fb0c2352c632@example.invalid,2025-08-10 13:05:11.126906+02,es,ES,True,17.82
1,1o1mni21lcmdr63fes,None,2025-08-03 20:31:40.887571+02,es,ES,False,0.00
2,1o1mni21lcmdscm0wx,None,2025-08-03 00:10:53.381649+02,es,RU,False,14.80
3,1o1mni21lcmdsee21z,anon+9b087c1df5b0@example.invalid,2025-08-02 20:52:23.190675+02,fr,FR,True,0.00
4,1o1mni21lcmdtiuw8t,anon+6c7e31760200@example.invalid,2025-08-04 18:41:04.23123+02,es,ES,True,NaN


In [28]:
sessions_dim = compute_conversion_flags(sessions_dim, order_header_df)
sessions_dim.head()

,sessionid,email,createdate,language,country,processed,total,converted
0,1o1mni21lcmdmzyowk,anon+fb0c2352c632@example.invalid,2025-08-10 13:05:11.126906+02,es,ES,True,17.82,True
1,1o1mni21lcmdr63fes,None,2025-08-03 20:31:40.887571+02,es,ES,False,0.00,False
2,1o1mni21lcmdscm0wx,None,2025-08-03 00:10:53.381649+02,es,RU,False,14.80,False
3,1o1mni21lcmdsee21z,anon+9b087c1df5b0@example.invalid,2025-08-02 20:52:23.190675+02,fr,FR,True,0.00,True
4,1o1mni21lcmdtiuw8t,anon+6c7e31760200@example.invalid,2025-08-04 18:41:04.23123+02,es,ES,True,NaN,False


***
# BUILD ATTRIBUTION

In [29]:
dm_prepared = prepare_dm_clicks(clicks_dm)

✓ DM clicks prepared
  Unique sessions with DM clicks: 164,775
  Total DM clicks: 213,884


In [30]:
dm_prepared.head()

,date,domain,sessionid,platform,url,ad,ad_type,ad_name,attr_value,attr_items,purchaseid,id,ad_country,ad_extension
175174,2024-12-04 16:35:44+01,originalpeople.com,1o1mni10k33m3schnr9,GA,/it/adesivi-personalizzati.html,it.shopping.car.display,shopping,car,NaN,NaN,NaN,696855,it,display
175169,2024-12-04 16:36:02+01,originalpeople.com,1o1mni10k33m3schnr9,GA,/it/adesivi-personalizzati.html,it.shopping.car.display,shopping,car,NaN,NaN,NaN,696860,it,display
174604,2024-12-05 20:59:25+01,originalpeople.com,1o1mni10k33m3vesc22,IGStore,/personalised-custom-car-stickers.html,sticker.wall,wall,NaN,NaN,NaN,NaN,697429,sticker,NaN
172876,2024-12-09 10:50:16+01,originalpeople.com,1o1mni10k33m3wrdwxi,FB,/fr/image-sticker/1,fr.x-mas24,x-mas24,NaN,144.54,"{""100010"": {""amount"": 17.4, ""quantity"": 1}, ""1...",19kmgm4d046e0,699170,fr,NaN
175660,2024-12-03 12:50:14+01,originalpeople.com,1o1mni14zo4m3z0r5e1,FB,/sv/personliga-stickers-bildekaler.html,se.carousel.camping,carousel,camping,NaN,NaN,NaN,696367,se,NaN


In [31]:
last_clicks = build_last_click_table(dm_prepared)

✓ Last-click table built
  Sessions with last click: 164,775


In [32]:
last_clicks.head()

,sessionid,last_click_date,ad,ad_type,ad_name,ad_country,platform
0,1o1mni10k33m3schnr9,2024-12-04 16:36:02+01,it.shopping.car.display,shopping,car,it,GA
1,1o1mni10k33m3vesc22,2024-12-05 20:59:25+01,sticker.wall,wall,None,sticker,IGStore
2,1o1mni10k33m3wrdwxi,2024-12-09 10:50:16+01,fr.x-mas24,x-mas24,None,fr,FB
3,1o1mni14zo4m3z0r5e1,2024-12-03 12:50:14+01,se.carousel.camping,carousel,camping,se,FB
4,1o1mni14zo4m40byn15,2024-12-03 20:09:45+01,de.carousel.caravan,carousel,caravan,de,FB


In [33]:
attribution = apply_attribution_window(order_header_df, last_clicks)

✓ Attribution window applied (7-day)
  Total orders: 132,961
  Orders with sessionid: 132,961
  Orders matched to DM clicks: 4,071
  Orders within 7-day window: 3,605
  Orders attributed to Organic/Direct: 129,356


In [34]:
attribution.tail()

,order_id,sessionid,createdate,last_click_date,ad,ad_type,ad_name,ad_country,platform,days_to_conversion,channel
132956,157393,1o1mni7n8qmezwmi0y,2025-09-01 07:49:37.827571,NaT,NaN,NaN,NaN,NaN,NaN,NaN,Organic/Direct
132957,157394,1o1mni7n8qmelhsin1,2025-09-01 09:41:22.378909,NaT,NaN,NaN,NaN,NaN,NaN,NaN,Organic/Direct
132958,157395,1o1mni7n8qmeya8kc0,2025-09-01 10:50:49.825583,NaT,NaN,NaN,NaN,NaN,NaN,NaN,Organic/Direct
132959,157396,1o1mni7n8qmf01e2d4,2025-09-01 11:13:36.195060,2025-08-31 18:43:07,de.shopping.caravan-feeds.caravan.de.1,shopping,caravan-feeds,de,GA,0.0,shopping
132960,157397,1o1mniiyxumf0zom9s,2025-09-01 11:46:04.791250,NaT,NaN,NaN,NaN,NaN,NaN,NaN,Organic/Direct


In [35]:
# Check for missing values in the whole dataframe
attribution.isnull().sum()

# turn to percentages
attribution.isnull().mean() * 100

order_id               0.000000
sessionid              0.000000
createdate             1.251495
last_click_date       96.938200
ad                    96.938200
ad_type               96.938200
ad_name               97.026948
ad_country            96.938200
platform              96.938200
days_to_conversion    96.999120
channel                0.000000
dtype: float64

In [36]:
attribution_final = finalize_attribution(attribution)

✓ Attribution finalized
  Unique sessions: 124,407

  Channel distribution:
    Organic/Direct: 120,876
    shopping: 2,231
    search: 431
    brevo: 211
    discovery: 199
    carousel: 173
    video: 120
    gifts: 39
    camper: 22
    car: 17


In [37]:
attribution_final.head()

,sessionid,channel,ad,ad_name,platform,ad_country
0,None,Organic/Direct,NaN,NaN,NaN,NaN
3538,oEGfUXO9MMV1VLlgffPd1sdbLTmISSSX,Organic/Direct,NaN,NaN,NaN,NaN
3539,UVv5ENvC0XPKFPwylx9IE4XyTx7PgU20,Organic/Direct,NaN,NaN,NaN,NaN
3540,OIHTuZ6Bd3yiNK9F1drRm1k53UXbuFQk,Organic/Direct,NaN,NaN,NaN,NaN
3541,K9Q6U2vZ2fszif2WDB20n8lNGPMPJiRi,Organic/Direct,NaN,NaN,NaN,NaN


In [38]:
attr_summary = attribution_summary(attribution_final, order_header_df)


📊 ATTRIBUTION SUMMARY:
   Total orders: 129,415
   Attributed to paid: 3,571 (2.8%)
   Organic/Direct: 125,844 (97.2%)
   Total revenue: €2,741,090.77


***
# Build Funnel

In [39]:
funnel_df = build_funnel(clicks_app, clicks_dm, order_header_df)
funnel_df.head()

,sessionid,clicks,channel,dm_clicks,orders,had_checkout_intent,converted
0,1o1mni10j3ym3n1rm1f,6,organic,0,0,0,0
1,1o1mni10k33m3n41wwx,37,organic,0,0,0,0
2,1o1mni10k33m3n42bxy,6,organic,0,0,0,0
3,1o1mni10k33m3n475r4,6,organic,0,0,0,0
4,1o1mni10k33m3n4e796,10,organic,0,0,0,0


***
# Exit Events

In [40]:
exit_events_df = extract_exit_events(clicks_app)
exit_events_df.head()

# Step 2: Identify which sessions exited without converting
exit_sessions_df = identify_exit_sessions(exit_events_df, funnel_df)
exit_sessions_df.head()

# Step 3: Get a summary of exit points
exit_summary = summarize_exit_points(exit_sessions_df, top_n=10)

✓ Extracted 163,028 exit events (last event per session)
  From 163,028 unique sessions
✓ Identified 148,575 exit sessions (sessions that didn't convert)
  Out of 148,575 total non-converted sessions

📊 EXIT POINTS SUMMARY:
   Total exit sessions: 148,575

   Top exit pages:
      1. stickers.html: 66,157
      2. newavatarbuilder.html: 29,245
      3. index.html: 14,362
      4. addon.html: 10,518
      5. cesta.html: 9,172

   Top exit event types:
      1. BUTTON: 56,789
      2. A: 46,078
      3. IMG: 30,758
      4. DIV: 7,367
      5. LI: 6,630


***
# Repeat Buyers Analysis

In [41]:
order_header_df = identify_customers(order_header_df, customer_col='cust_email')

# Step 2: Classify orders as first-time or repeat
order_header_df = classify_orders(order_header_df)

# Check the classification
order_header_df[['order_id', 'customer_id', 'createdate', 'first_purchase_date',
                 'order_number', 'is_first_purchase', 'is_repeat_purchase']].head(10)

✓ Customer identification complete
  Customer proxy: cust_email
  Total orders: 132,961
  Orders with customer_id: 132,961 (100.0%)
  Unique customers: 108,549
  Avg orders per customer: 1.22
  ⚠️  Warning: 1,664 orders have null createdate - excluding from classification

✓ Orders classified
  Valid orders for classification: 131,297
  First-time orders: 107,361 (80.7%)
  Repeat orders: 23,936 (18.0%)
  Customers with 2+ orders: 17,503
  Max orders by single customer: 185.0


,order_id,customer_id,createdate,first_purchase_date,order_number,is_first_purchase,is_repeat_purchase
0,1160,anon+9a697ab42d37@example.invalid,2016-09-26 13:48:37.710340,2016-09-26 13:48:37.710340,1.0,True,False
1,1161,anon+9a697ab42d37@example.invalid,2016-09-26 13:51:47.935259,2016-09-26 13:48:37.710340,2.0,False,True
2,1162,anon+bb33103e784d@example.invalid,2016-09-26 18:19:39.710063,2016-09-26 18:19:39.710063,1.0,True,False
3,1163,anon+bb33103e784d@example.invalid,2016-09-26 18:25:35.678834,2016-09-26 18:19:39.710063,2.0,False,True
4,1164,anon+7f013fb95025@example.invalid,2016-09-26 18:26:53.532779,2016-09-26 18:26:53.532779,1.0,True,False
5,1165,anon+bb33103e784d@example.invalid,2016-09-26 18:31:58.404392,2016-09-26 18:19:39.710063,3.0,False,True
6,1166,anon+06c76d93f202@example.invalid,2016-09-26 19:14:45.317333,2016-09-26 19:14:45.317333,1.0,True,False
7,1167,anon+06c76d93f202@example.invalid,2016-09-26 19:15:09.789151,2016-09-26 19:14:45.317333,2.0,False,True
8,1168,anon+06c76d93f202@example.invalid,2016-09-26 19:18:42.908318,2016-09-26 19:14:45.317333,3.0,False,True
9,1169,anon+06c76d93f202@example.invalid,2016-09-26 19:21:50.734063,2016-09-26 19:14:45.317333,4.0,False,True


In [42]:
# Step 3: Calculate repeat buyer metrics
repeat_metrics = calculate_repeat_metrics(order_header_df)

# Step 4: Build cohort analysis (3 and 6 month windows)
cohort_df = build_cohort_analysis(order_header_df, windows=[3, 6])
cohort_df.head(10)


📊 REPEAT BUYER METRICS:
   Total orders: 132,961
   First-time orders: 107,361 (80.7%)
   Repeat orders: 23,936 (18.0%)

   Unique customers: 108,549
   Repeat customers: 17,503 (16.1%)

   First-time revenue: €2,251,576.03 (80.6%)
   Repeat revenue: €501,354.29 (17.9%)

   First-time AOV: €20.97
   Repeat AOV: €20.95

✓ Cohort analysis built
  Cohorts (months): 109
  Date range: 2016-09 to 2025-09
  Retention windows: [3, 6] months


,cohort_month,customers,first_purchase_revenue,repeat_within_3m,repeat_rate_3m_%,repeat_within_6m,repeat_rate_6m_%
0,2016-09,47,702.715,6,12.77,7,14.89
1,2016-10,266,3963.850,20,7.52,20,7.52
2,2016-11,315,4755.580,25,7.94,29,9.21
3,2016-12,391,6121.385,24,6.14,31,7.93
4,2017-01,271,4303.975,22,8.12,25,9.23
5,2017-02,298,4330.410,24,8.05,30,10.07
6,2017-03,461,6879.995,28,6.07,32,6.94
7,2017-04,299,4285.705,33,11.04,39,13.04
8,2017-05,352,5204.615,16,4.55,17,4.83
9,2017-06,353,5333.385,18,5.10,22,6.23


***
# EXPORT EVERYTHING

In [43]:
# EXPORT EVERYTHING
export_tables(
    tables={
        'order_header': order_header_df,  
        'line_items': order_line_items_df,
        'sessions': sessions_dim,
        'attribution': attribution_final,
        'funnel': funnel_df,
        'exit_events': exit_events_df,
        'exit_sessions': exit_sessions_df,
        'cohorts': cohort_df 
    }
)

✓ Exported order_header.csv
✓ Exported line_items.csv
✓ Exported sessions.csv
✓ Exported attribution.csv
✓ Exported funnel.csv
✓ Exported exit_events.csv
✓ Exported exit_sessions.csv
✓ Exported cohorts.csv

✓ All 8 tables exported to '/Users/borisivanov/Documents/GitHub/sales-management-project-op-analytics/data/processed_data/'
